# Work across Excel and JSON formats · Student

Pandas, Clearly · Pandas for Data Science · Version 1.0.0

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates

Read, predict, run, change, and explain. Every worked example starts from fresh teaching data. The original student/café tables are fictional; the public Penguins files retain their documented source.

Run the setup cell first. Keep this notebook inside the downloaded course folder so its `data/` folder can be found. Exports go to `outputs/`. For public-data lessons, extract the full notebook practice pack.

Questions: What does one row represent? Which files are inputs, and where will your outputs be written?

In [ ]:
# Pandas, Clearly
# Dr. Ali Al Bataineh
# Computer Information Science
# Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates
# Run once before the lessons. It prepares explicitly fictional practice data.
# Questions: What does 'as pd' mean? Where will exported files be saved?
# Try: print(pd.__version__) and print(OUTPUT_DIR.resolve()).
# Explain: Why should source data and analysis outputs use different folders?
import os
import tempfile
from pathlib import Path
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "pandas-clearly-mpl"))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import sqlite3
from pathlib import Path
from io import StringIO

# Find data from the repository root or either notebook subdirectory.
# Browser workers mount the same data tree at /course/data.
COURSE_ROOT = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents]
     if (parent / "data" / "students.csv").is_file()),
    Path.cwd(),
)
DATA_DIR = COURSE_ROOT / "data"
OUTPUT_DIR = COURSE_ROOT / "outputs"
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

_STUDENT_ROWS = [
    ["S001", "Amira", "RAK", "AI", 20, 8, 88, 95, "2026-09-01"],
    ["S002", "Bilal", "Dubai", "IT", 21, 5, 72, 85, "2026-09-02"],
    ["S003", "Chloe", "RAK", "AI", 19, 9, 91, 98, "2026-09-03"],
    ["S004", "Dana", "Dubai", "IT", 22, 4, 65, 78, "2026-09-04"],
    ["S005", "Ethan", "RAK", "IT", 20, 6, 79, 90, "2026-09-05"],
    ["S006", "Fatima", "Dubai", "AI", 21, 10, 95, 99, "2026-09-06"],
    ["S007", "Grace", "RAK", "AI", 23, 7, 84, 92, "2026-09-07"],
    ["S008", "Hassan", "Dubai", "IT", 20, 3, 58, 70, "2026-09-08"],
    ["S009", "Ivy", "RAK", "IT", 22, 6, 76, 88, "2026-09-09"],
    ["S010", "Jamal", "Dubai", "AI", 19, 8, 89, 96, "2026-09-10"],
    ["S011", "Khalid", "RAK", "AI", 24, 5, 68, 82, "2026-09-11"],
    ["S012", "Lina", "Dubai", "IT", 21, 7, 82, 94, "2026-09-12"],
]
_STUDENT_COLUMNS = ["Student_ID", "Name", "Campus", "Program", "Age",
                    "Study_Hours", "Score", "Attendance", "Join_Date"]

def students():
    """Return a new, clean practice DataFrame, never real student records."""
    return pd.DataFrame(_STUDENT_ROWS, columns=_STUDENT_COLUMNS)

def records():
    """Return the same practice records as a list of dictionaries."""
    return students().to_dict(orient="records")

def raw_students():
    """Return a fresh table with deliberate data-quality problems."""
    df = students()
    df["Score"] = df["Score"].astype(object)
    df["Study_Hours"] = df["Study_Hours"].astype(object)
    df.loc[0, "Name"] = " Amira "
    df.loc[1, "Campus"] = "dubai"
    df.loc[3, "Score"] = None
    df.loc[5, "Study_Hours"] = None
    df.loc[7, "Score"] = "absent"
    df.loc[10, "Attendance"] = 105
    return pd.concat([df, df.iloc[[2]]], ignore_index=True)

def departments():
    return pd.DataFrame({
        "Program": ["AI", "IT"],
        "Department": ["Computing", "Information Systems"],
        "Advisor": ["Dr. Noor", "Dr. Sami"],
    })

def assessment():
    return pd.DataFrame({
        "Student_ID": ["S001", "S002", "S003", "S004", "S005", "S006",
                       "S007", "S008", "S009", "S010", "S011", "S013"],
        "Quiz": [17, 14, 19, 13, 16, 19, 17, 11, 15, 18, 14, 16],
        "Project": [44, 36, 46, 34, 40, 47, 42, 30, 38, 45, 35, 41],
    })

def attendance():
    return pd.DataFrame({
        "Date": ["2026-10-01", "2026-10-02", "2026-10-03",
                 "2026-10-05", "2026-10-06", "2026-10-07"],
        "Visitors": [20, 24, 18, 30, 28, 32],
        "Campus": ["RAK"] * 6,
    })

# Create missing practice files. Existing files are preserved.
for _name, _frame in [
    ("students.csv", students()), ("raw_students.csv", raw_students()),
    ("departments.csv", departments()), ("assessment.csv", assessment()),
    ("attendance.csv", attendance()),
]:
    if not (DATA_DIR / _name).exists():
        _frame.to_csv(DATA_DIR / _name, index=False)
if not (DATA_DIR / "students.json").exists():
    students().to_json(DATA_DIR / "students.json", orient="records", indent=2)
if not (DATA_DIR / "students.xlsx").exists():
    try:
        students().to_excel(DATA_DIR / "students.xlsx", sheet_name="Students", index=False)
    except ImportError:
        print("Excel needs openpyxl. Install it before Module 20's Excel cell.")

pd.set_option("display.max_rows", 15)
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 110)
plt.rcParams.update({"figure.figsize": (7, 4), "axes.spines.top": False,
                     "axes.spines.right": False, "figure.dpi": 110})
print("Ready. pandas", pd.__version__)
print("Practice data: 12 fictional students. Every helper returns a fresh table.")
print("Exports go to:", OUTPUT_DIR.resolve())


## Module 28. Work across Excel and JSON formats

Read worksheets and JSON structures without changing the underlying observations.

Before you start: You can read files, interpret missing values, and compare DataFrames.

Official reference: https://pandas.pydata.org/docs/user_guide/io.html

### M28C01. Choose the worksheet you actually need

An Excel workbook can contain both observations and documentation. The bundled workbook has a Penguins data sheet and a Data_Dictionary sheet with field meanings and units. Specify the intended sheet so a change in tab order does not silently change your input.

**Read:** Why specify sheet_name="Penguins"?

Your explanation: _write here_.

**Predict:** Will the data sheet have the same 344 rows as the source CSV?

Your explanation: _write here_.

**Explain:** Why should a data dictionary be kept separate from the observation rows?

Your explanation: _write here_.



In [ ]:
# Predict before running: Will the data sheet have the same 344 rows as the source CSV?
path = DATA_DIR / "public" / "penguins.xlsx"
with pd.ExcelFile(path, engine="openpyxl") as book:
    print("Worksheets:", book.sheet_names)
    result = pd.read_excel(book, sheet_name="Penguins")
print(result.head(3))
print("Shape:", result.shape)
print("Known body masses:", result["body_mass_g"].count())

**Your turn:** Read only A:D from Data_Dictionary into a DataFrame named dictionary. Select the fields measured in mm or g into result and print those rows.

For the check, keep these result names: `dictionary`, `result`.

**Hint:** Use usecols="A:D", then filter unit with isin(["mm", "g"]).

**Watch for this:** The Excel lesson requires openpyxl. A .xlsx file is not plain delimited text and should not be passed to read_csv.

In [ ]:
# Change this code to complete the task.
# Question: Read only A:D from Data_Dictionary into a DataFrame named dictionary. Select the fields measured in mm or g into result and print those rows.
path = DATA_DIR / "public" / "penguins.xlsx"
with pd.ExcelFile(path, engine="openpyxl") as book:
    print("Worksheets:", book.sheet_names)
    result = pd.read_excel(book, sheet_name="Penguins")
print(result.head(3))
print("Shape:", result.shape)
print("Known body masses:", result["body_mass_g"].count())

**Quick check:** Which argument selects an Excel worksheet by name?

1. sheet_name
2. orient
3. lines

**Explain what you learned:** Inspect workbook structure and choose a worksheet explicitly.

### M28C02. Read an array of JSON records

Records-oriented JSON represents each row as an object whose keys name fields. The bundled JSON is a documented conversion of the original public CSV. Its null values represent the same missing entries as the source NA tokens.

**Read:** What corresponds to one table row in this JSON structure?

Your explanation: _write here_.

**Predict:** Will JSON null become the text string "null" in the DataFrame?

Your explanation: _write here_.

**Explain:** Why can equivalent CSV and JSON imports have different inferred pandas dtypes?

Your explanation: _write here_.



In [ ]:
# Predict before running: Will JSON null become the text string "null" in the DataFrame?
path = DATA_DIR / "public" / "penguins.records.json"
df = pd.read_json(path, orient="records")
result = df[["species", "island", "body_mass_g"]].head(5)
print(result)
print("Loaded records:", len(df))
print("Missing body masses:", df["body_mass_g"].isna().sum())

**Your turn:** Read the records JSON and summarize the number of observations and known body masses by species. Store the table in result with columns species, Observations, and Known_mass; keep species as a normal column.

For the check, keep these result names: `result`.

**Hint:** Use groupby(..., as_index=False) with size and count.

**Watch for this:** JSON orientation describes structure. Use the orientation that matches the supplied document.

In [ ]:
# Change this code to complete the task.
# Question: Read the records JSON and summarize the number of observations and known body masses by species. Store the table in result with columns species, Observations, and Known_mass; keep species as a normal column.
path = DATA_DIR / "public" / "penguins.records.json"
df = pd.read_json(path, orient="records")
result = df[["species", "island", "body_mass_g"]].head(5)
print(result)
print("Loaded records:", len(df))
print("Missing body masses:", df["body_mass_g"].isna().sum())

**Quick check:** In records-oriented JSON, field names are stored as what?

1. Object keys
2. Excel worksheet tabs
3. Pandas row positions

**Explain what you learned:** Read JSON according to its actual record structure.

### M28C03. Read JSON Lines in manageable batches

JSON Lines stores one complete JSON object on each line, without an outer array. Use lines=True so pandas reads that structure correctly. Batch processing can summarize a larger file without retaining every row at once, provided you combine the statistics correctly.

**Read:** What tells pandas that each line is a separate JSON record?

Your explanation: _write here_.

**Predict:** How many records should this JSON Lines file contain?

Your explanation: _write here_.

**Explain:** Why is the average of chunk means generally not the overall mean?

Your explanation: _write here_.



In [ ]:
# Predict before running: How many records should this JSON Lines file contain?
path = DATA_DIR / "public" / "penguins.jsonl"
df = pd.read_json(path, lines=True)
result = df.head(3)
print(result)
print("Rows:", len(df))

**Your turn:** Read the file in chunks of 100 rows. Accumulate total rows, the sum of known body masses, and the count of known body masses. Store a dictionary named result with keys rows, mass_sum_g, known_mass, and mean_mass_g for those totals and the overall mean.

For the check, keep these result names: `result`.

**Hint:** Add each chunk sum and count separately, then divide the combined sum by the combined known count.

**Watch for this:** This file is small; chunking is taught as a technique, not claimed as a speed improvement here.

In [ ]:
# Change this code to complete the task.
# Question: Read the file in chunks of 100 rows. Accumulate total rows, the sum of known body masses, and the count of known body masses. Store a dictionary named result with keys rows, mass_sum_g, known_mass, and mean_mass_g for those totals and the overall mean.
path = DATA_DIR / "public" / "penguins.jsonl"
df = pd.read_json(path, lines=True)
result = df.head(3)
print(result)
print("Rows:", len(df))

**Quick check:** Which calculation correctly combines unequal chunks?

1. The unweighted average of their means
2. Total known-value sum divided by total known-value count
3. The largest chunk mean

**Explain what you learned:** Combine batch statistics using the matching totals and counts.

### M28C04. Flatten nested JSON and verify the values

Nested JSON can place related fields inside another object. The teaching version groups the four measurements under measurements while preserving all source values. json_normalize flattens those nested fields into columns with dotted names.

**Read:** What does measurements.body_mass_g describe in the flattened table?

Your explanation: _write here_.

**Predict:** Does flattening change the number of penguin observations?

Your explanation: _write here_.

**Explain:** Why should you check row counts and values after flattening a less familiar JSON document?

Your explanation: _write here_.



In [ ]:
# Predict before running: Does flattening change the number of penguin observations?
path = DATA_DIR / "public" / "penguins.nested.json"
payload = json.loads(path.read_text(encoding="utf-8"))
result = pd.json_normalize(payload, sep=".")
print(result.head(3))
print("Columns:", result.columns.tolist())
print("Known body masses:", result["measurements.body_mass_g"].count())

**Your turn:** Flatten the file, remove the measurements. prefix from column names, and restore the CSV column order. Store the source CSV in reference and the flattened table in result, using the same explicit nullable schema for both. Compare all values and missing entries.

For the check, keep these result names: `reference`, `result`.

**Hint:** Use rename for the four measurement columns, reindex columns from the reference, then astype with the shared schema.

**Watch for this:** The nested structure is a course-created representation, not a claim about an upstream research API.

In [ ]:
# Change this code to complete the task.
# Question: Flatten the file, remove the measurements. prefix from column names, and restore the CSV column order. Store the source CSV in reference and the flattened table in result, using the same explicit nullable schema for both. Compare all values and missing entries.
path = DATA_DIR / "public" / "penguins.nested.json"
payload = json.loads(path.read_text(encoding="utf-8"))
result = pd.json_normalize(payload, sep=".")
print(result.head(3))
print("Columns:", result.columns.tolist())
print("Known body masses:", result["measurements.body_mass_g"].count())

**Quick check:** Does json_normalize recover missing research measurements?

1. Yes, from the field names
2. No, it changes the structure while preserving missingness
3. Only for nested numbers

**Explain what you learned:** Verify semantic equality after changing a data representation.

## Keep practicing

Can you apply this idea to a different table without copying the example?

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates